<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/domain_analyze.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import difflib
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---------------------------------------------------------------------------
# 1. 행정구역 약칭 -> 정식명칭 매핑
# ---------------------------------------------------------------------------

SIDO_ABBR_TO_OFFICIAL = {
    "서울": "서울특별시",
    "부산": "부산광역시",
    "대구": "대구광역시",
    "인천": "인천광역시",
    "광주": "광주광역시",
    "대전": "대전광역시",
    "울산": "울산광역시",
    "세종": "세종특별자치시",
    "경기": "경기도",
    "강원": "강원특별자치도",
    "충북": "충청북도",
    "충남": "충청남도",
    "전북": "전북특별자치도",
    "전남": "전라남도",
    "경북": "경상북도",
    "경남": "경상남도",
    "제주": "제주특별자치도",
}

# 창원처럼 시도 자리에 실제로는 시/군 이름이 오는 예외 케이스.
# 값은 (정식 시도명, 보정된 sigungu) — 사건 데이터의 sigungu 토큰과 합쳐서 사용.
SPECIAL_SIDO_ALIASES = {
    "창원": "경상남도",
}


def normalize_sido(sido_raw: str) -> dict:

    if not isinstance(sido_raw, str) or not sido_raw.strip():
        return {"sido_official": None, "is_matched": False, "note": "missing"}

    key = sido_raw.strip()

    if key in SIDO_ABBR_TO_OFFICIAL:
        return {"sido_official": SIDO_ABBR_TO_OFFICIAL[key], "is_matched": True, "note": None}

    if key in SPECIAL_SIDO_ALIASES:
        return {
            "sido_official": SPECIAL_SIDO_ALIASES[key],
            "is_matched": True,
            "note": f"'{key}'는 특례시 표기 - sigungu를 '{key}시'로 보정 필요",
        }

    # 매핑 테이블에 없는 값 (오탈자, 신규 행정구역 명칭 등)
    return {"sido_official": None, "is_matched": False, "note": f"미등록 시도명: {key}"}


def normalize_fire_events_region(events_df: pd.DataFrame) -> pd.DataFrame:
    """
    소방청 사건 DataFrame(sido/sigungu/emd 컬럼 보유)에
    sido_official, region_match_note 컬럼을 추가한다.
    """
    out = events_df.copy()
    normalized = out["sido"].apply(normalize_sido)

    out["sido_official"] = normalized.apply(lambda d: d["sido_official"])
    out["sido_matched"] = normalized.apply(lambda d: d["is_matched"])
    out["region_note"] = normalized.apply(lambda d: d["note"])

    # 창원처럼 시도 자리에 도시명이 온 경우, sigungu를 '창원시'로 보정
    special_mask = out["sido"].isin(SPECIAL_SIDO_ALIASES.keys())
    out.loc[special_mask, "sigungu"] = out.loc[special_mask, "sido"] + "시"

    return out


# ---------------------------------------------------------------------------
# 2. URL 도메인 신뢰도 분석
# ---------------------------------------------------------------------------
# 실제 공식 재난 문자 발신 도메인 목록. 프로젝트에서 사용할 정확한 리스트는
# 공공데이터포털/각 기관 공식 발표 자료로 교체 필요 — 아래는 검증용 예시.
OFFICIAL_DOMAINS = {
    "safekorea.go.kr",
    "korea119.go.kr",
    "nfa.go.kr",
    "weather.go.kr",
    "kma.go.kr",
    "vo.la",
}

# 대한민국 정부기관은 공통적으로 .go.kr TLD를 사용하므로
# (예: kma.go.kr, cbs042.kma.go.kr 등 하위 도메인 포함) 개별 화이트리스트
# 등록 여부와 무관하게 .go.kr로 끝나면 공식 도메인으로 우선 인정한다.
OFFICIAL_TLD_SUFFIXES = (".go.kr",)

SHORTENER_DOMAINS = {
    "vo.la", "url.kr", "bit.ly", "han.gl", "t.ly", "goo.gl", "tinyurl.com",
    "t.co", "ow.ly", "is.gd", "buff.ly", "cutt.ly", "rb.gy",
}

RISKY_TLDS = {".tk", ".ml", ".ga", ".cf", ".gq", ".xyz", ".top", ".buzz"}


def is_official_domain(domain: str) -> bool:
    if any(domain == d or domain.endswith("." + d) for d in OFFICIAL_DOMAINS):
        return True
    return any(domain.endswith(suffix) for suffix in OFFICIAL_TLD_SUFFIXES)


def typo_similarity_to_official(domain: str) -> tuple[float, str]:
    """공식 도메인 목록과의 최대 유사도(0~1)와 가장 유사한 도메인을 반환."""
    best_score, best_domain = 0.0, None
    for official in OFFICIAL_DOMAINS:
        score = difflib.SequenceMatcher(None, domain, official).ratio()
        if score > best_score:
            best_score, best_domain = score, official
    return round(best_score, 3), best_domain


def domain_risk_score(domain: str) -> dict:

    if not isinstance(domain, str) or not domain:
        return {}

    domain = domain.lower().strip()

    if is_official_domain(domain):
        return {
            "domain": domain, "is_official": True, "is_shortened": False,
            "has_risky_tld": False, "typo_similarity": 1.0, "typo_similar_to": domain,
            "contains_ip": False, "risk_score": 0.0,
        }

    is_shortened = domain in SHORTENER_DOMAINS
    has_risky_tld = any(domain.endswith(tld) for tld in RISKY_TLDS)
    contains_ip = bool(re.match(r'^\d{1,3}(\.\d{1,3}){3}$', domain))
    typo_sim, typo_target = typo_similarity_to_official(domain)

    risk = 0.0
    risk += 0.3 if is_shortened else 0       # 단축 URL: 원본 도메인이 가려짐
    risk += 0.2 if has_risky_tld else 0      # 스팸/피싱에 흔히 쓰이는 TLD
    risk += 0.3 if contains_ip else 0        # 도메인 대신 IP 직접 노출
    risk += 0.2 * typo_sim                    # 공식 도메인과 유사할수록 타이포스쿼팅 의심

    return {
        "domain": domain,
        "is_official": False,
        "is_shortened": is_shortened,
        "has_risky_tld": has_risky_tld,
        "typo_similarity": typo_sim,
        "typo_similar_to": typo_target,
        "contains_ip": contains_ip,
        "risk_score": round(min(risk, 1.0), 3),
    }


def analyze_sms_url_domains(sms_df: pd.DataFrame, domain_col: str = "url_domains") -> pd.DataFrame:

    out = sms_df.copy()

    def evaluate_row(domains):
        if not isinstance(domains, list) or len(domains) == 0:
            return {"max_domain_risk": 0.0, "domain_risk_details": []}
        details = [domain_risk_score(d) for d in domains]
        max_risk = max(d["risk_score"] for d in details)
        return {"max_domain_risk": max_risk, "domain_risk_details": details}

    evaluated = out[domain_col].apply(evaluate_row)
    out["max_domain_risk"] = evaluated.apply(lambda d: d["max_domain_risk"])
    out["domain_risk_details"] = evaluated.apply(lambda d: d["domain_risk_details"])
    return out


# ---------------------------------------------------------------------------
# 실행 예시 (모듈 단독 실행 시 검증용)
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    import ast

    sms = pd.read_csv("/content/drive/MyDrive/REDRED/sms_url_enriched.csv")
    # CSV에서 리스트 컬럼은 문자열로 저장되므로 다시 파싱
    sms["url_domains"] = sms["url_domains"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
    )

    print("=== URL 도메인 위험도 분석 ===")
    sms_analyzed = analyze_sms_url_domains(sms)
    url_rows = sms_analyzed[sms_analyzed["url_domains"].apply(len) > 0]
    print(url_rows[["message", "url_domains", "max_domain_risk"]].head(10).to_string())

    print("\n=== 행정구역 정규화 (소방청 사건 데이터 재구성 후 테스트) ===")
    # regex_module_fixed.py 로직으로 사건 단위 재구성 (region_note 검증용)
    fire = pd.read_csv("/content/drive/MyDrive/REDRED/소방청일일상황보고.csv")
    event_pattern = re.compile(r'❍\s*\((.*?)\)')
    region_pattern = re.compile(r'([가-힣]{2}\s?\S*(?:시|군|구)?\s?\S*(?:동|읍|면|리)?)\s*『')

    rows = []
    for _, row in fire.iterrows():
        text = str(row["clean_text"])
        matches = list(event_pattern.finditer(text))
        for i, m in enumerate(matches):
            start = m.start()
            end = matches[i + 1].start() if i < len(matches) - 1 else len(text)
            block = text[start:end]
            rmatch = region_pattern.search(block)
            region_raw = rmatch.group(1).strip() if rmatch else None
            tokens = region_raw.split() if region_raw else []
            rows.append({
                "sido": tokens[0] if len(tokens) > 0 else None,
                "sigungu": tokens[1] if len(tokens) > 1 else None,
                "emd": tokens[2] if len(tokens) > 2 else None,
            })

    events_df = pd.DataFrame(rows)
    events_normalized = normalize_fire_events_region(events_df)

    print("전체 사건 수:", len(events_normalized))
    print("정식명칭 매핑 성공:", events_normalized["sido_matched"].sum())
    print("매핑 실패(미등록 시도명) 케이스:")
    print(events_normalized[~events_normalized["sido_matched"]]["sido"].value_counts())

    events_normalized.to_csv("/content/drive/MyDrive/REDRED/fire_events_region_normalized.csv", index=False)
    sms_analyzed.to_csv("/content/drive/MyDrive/REDRED/sms_url_risk_analyzed.csv", index=False)
    print("\n저장 완료")


=== URL 도메인 위험도 분석 ===
                                                                                                                                                           message  url_domains  max_domain_risk
0                                                                             오늘 05:50 미평천 청주시(장성2교)지점 심각단계. 하천범람에 대비 바랍니다. 내 위치, 침수우려지역 확인 vo.la/Vo27SU [금강홍수통제소]      [vo.la]              0.0
12                                                                               오늘 06:00 용수천 세종시(도암교)지점 홍수경보. 피해에 대비 바랍니다. 내 위치, 침수우려지역 확인 vo.la/0Pv3TX [금강홍수통제소]      [vo.la]              0.0
51                                                                            오늘 12:10 도림천 서울시(신대방1교)지점 홍수주의보. 피해에 대비 바랍니다. 내 위치, 침수우려지역 확인 vo.la/Y8XmTU [한강홍수통제소]      [vo.la]              0.0
114                                                                        06:39 세종특별자치시 장군면 인근 50mm/h 이상 강한 비로 침수 등 우려(국민행동요령: cbs042.kma.go.kr) Heavy rain [기상청]  [kma.go.kr]              0.0
115         